# PyTorch Lightning classifiers for the SHAP study

ResNet-based classifiers (torchvision ResNet adapted to MNIST and CIFAR-10) trained with **PyTorch Lightning** and evaluated with scikit-learn's `classification_report`. These models were used alongside the Keras models in the Kernel SHAP group project.

In [ ]:
#GPU count and name
!nvidia-smi -L

In [ ]:
import torch
torch.__version__

In [ ]:
!pip install ipython-autotime

%load_ext autotime

### MNIST

In [97]:
from torchvision.models import resnet18
from torch import nn
from torch.utils.data import DataLoader

time: 1.3 ms (started: 2021-06-29 16:04:42 +00:00)


Data


In [5]:
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor

In [ ]:
train_ds = MNIST("mnist", train=True, download=True, transform=ToTensor())
test_ds = MNIST("mnist", train=False, download=True, transform=ToTensor())

In [9]:
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64)

Model

In [ ]:
!pip install git+https://github.com/PyTorchLightning/pytorch-lightning
import pytorch_lightning as pl
print(pl.__version__)

In [13]:
import pytorch_lightning as pl
from pytorch_lightning.core.decorators import auto_move_data

In [ ]:
class ResNetMNIST(pl.LightningModule):
  def __init__(self):
    super().__init__()
    self.model = resnet18(num_classes=10)
    self.model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    self.loss = nn.CrossEntropyLoss()

  @auto_move_data
  def forward(self, x):
    return self.model(x)
  
  def training_step(self, batch, batch_no):
    x, y = batch
    logits = self(x)
    loss = self.loss(logits, y)
    return loss
  
  def configure_optimizers(self):
    return torch.optim.RMSprop(self.parameters(), lr=0.005)

In [15]:
model = ResNetMNIST()

Train

In [16]:
trainer = pl.Trainer(
    gpus=1,
    max_epochs=1,
    progress_bar_refresh_rate=20
)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs


In [17]:
trainer.fit(model, train_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type             | Params
-------------------------------------------
0 | model | ResNet           | 11.2 M
1 | loss  | CrossEntropyLoss | 0     
-------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.701    Total estimated model params size (MB)


/usr/local/lib/python3.7/dist-packages/torch/nn/functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  /pytorch/c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


In [18]:
trainer.save_checkpoint("resnet18_mnist.pt")

Test


In [19]:
def get_prediction(x, model: pl.LightningModule):
  model.freeze() # prepares model for predicting
  probabilities = torch.softmax(model(x), dim=1)
  predicted_class = torch.argmax(probabilities, dim=1)
  return predicted_class, probabilities

In [20]:
from tqdm.autonotebook import tqdm

In [21]:
inference_model = ResNetMNIST.load_from_checkpoint("resnet18_mnist.pt", map_location="cuda")

In [22]:
true_y, pred_y = [], []
for batch in tqdm(iter(test_dl), total=len(test_dl)):
  x, y = batch
  true_y.extend(y)
  preds, probs = get_prediction(x, inference_model)
  pred_y.extend(preds.cpu())

Results

In [23]:
from sklearn.metrics import classification_report

In [24]:
print(classification_report(true_y, pred_y, digits=3))

              precision    recall  f1-score   support

           0      0.996     0.958     0.977       980
           1      0.983     0.991     0.987      1135
           2      0.987     0.756     0.856      1032
           3      0.652     0.995     0.788      1010
           4      0.997     0.974     0.985       982
           5      0.964     0.978     0.971       892
           6      0.980     0.987     0.984       958
           7      0.985     0.968     0.976      1028
           8      0.989     0.718     0.832       974
           9      0.937     0.962     0.950      1009

    accuracy                          0.929     10000
   macro avg      0.947     0.929     0.931     10000
weighted avg      0.947     0.929     0.930     10000



## Cifar10

Data

In [52]:
from torchvision.datasets import CIFAR10

In [27]:
train_ds = CIFAR10("cifar10", train=True, download=True, transform=ToTensor())
test_ds = CIFAR10("cifar10", train=False, download=True, transform=ToTensor())


Extracting cifar10/cifar-10-python.tar.gz to cifar10
Files already downloaded and verified


In [53]:
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=64)

Model

In [98]:
from torchvision.models import resnet152

time: 966 µs (started: 2021-06-29 16:04:55 +00:00)


In [99]:
class ResNetCIFAR10(pl.LightningModule):
  def __init__(self):
    super().__init__()
    self.model = resnet152(num_classes=10)
    self.loss = nn.CrossEntropyLoss()

  @auto_move_data
  def forward(self, x):
    return self.model(x)
  
  def training_step(self, batch, batch_no):
    x, y = batch
    logits = self(x)
    loss = self.loss(logits, y)
    return loss
  
  def configure_optimizers(self):
    return torch.optim.RMSprop(self.parameters(), lr=0.005)

time: 8.88 ms (started: 2021-06-29 16:04:56 +00:00)


/usr/local/lib/python3.7/dist-packages/pytorch_lightning/core/decorators.py:65: LightningDeprecationWarning: The `@auto_move_data` decorator is deprecated in v1.3 and will be removed in v1.5. Please use `trainer.predict` instead for inference. The decorator was applied to `forward`
  "The `@auto_move_data` decorator is deprecated in v1.3 and will be removed in v1.5."


In [100]:
model = ResNetCIFAR10()

time: 833 ms (started: 2021-06-29 16:04:58 +00:00)


Train

In [102]:
trainer = pl.Trainer(
    gpus=1,
    max_epochs=30,
    progress_bar_refresh_rate=20
)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs


time: 8.77 ms (started: 2021-06-29 16:05:10 +00:00)


In [103]:
trainer.fit(model, train_dl)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type             | Params
-------------------------------------------
0 | model | ResNet           | 58.2 M
1 | loss  | CrossEntropyLoss | 0     
-------------------------------------------
58.2 M    Trainable params
0         Non-trainable params
58.2 M    Total params
232.657   Total estimated model params size (MB)



time: 51min 15s (started: 2021-06-29 16:05:11 +00:00)


In [104]:
trainer.save_checkpoint("resnet34_cifar10.pt")

time: 2.56 s (started: 2021-06-29 16:56:27 +00:00)


Test

In [105]:
def get_prediction(x, model: pl.LightningModule):
  model.freeze() # prepares model for predicting
  probabilities = torch.softmax(model(x), dim=1)
  predicted_class = torch.argmax(probabilities, dim=1)
  return predicted_class, probabilities

time: 3.74 ms (started: 2021-06-29 16:56:30 +00:00)


In [106]:
from tqdm.autonotebook import tqdm

time: 977 µs (started: 2021-06-29 16:56:30 +00:00)


In [107]:
inference_model = ResNetCIFAR10.load_from_checkpoint("resnet34_cifar10.pt", map_location="cuda")

time: 1.57 s (started: 2021-06-29 16:56:30 +00:00)


In [108]:
true_y, pred_y = [], []
for batch in tqdm(iter(test_dl), total=len(test_dl)):
  x, y = batch
  true_y.extend(y)
  preds, probs = get_prediction(x, inference_model)
  pred_y.extend(preds.cpu())


time: 2min 6s (started: 2021-06-29 16:56:31 +00:00)


Results

In [109]:
from sklearn.metrics import classification_report

time: 1.11 ms (started: 2021-06-29 16:58:38 +00:00)


In [110]:
print(classification_report(true_y, pred_y, digits=3))

              precision    recall  f1-score   support

           0      0.706     0.834     0.765      1000
           1      0.739     0.894     0.809      1000
           2      0.700     0.610     0.652      1000
           3      0.545     0.526     0.535      1000
           4      0.709     0.611     0.656      1000
           5      0.555     0.671     0.608      1000
           6      0.749     0.800     0.774      1000
           7      0.699     0.840     0.763      1000
           8      0.900     0.776     0.834      1000
           9      0.926     0.529     0.673      1000

    accuracy                          0.709     10000
   macro avg      0.723     0.709     0.707     10000
weighted avg      0.723     0.709     0.707     10000

time: 194 ms (started: 2021-06-29 16:58:38 +00:00)


In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')